# `%catalog` smoke test — immediate execution, no commit

A functional test notebook for the `%catalog` magic (`src/eea_datalakehouse/notebook/magics.py`),
meant to be run against this project's own debug/local Dremio stack (`debugger/.env` — see
`debugger/debug_run.py`, whose `run_createfolder`/`run_setmeta2wiki`/... functions exercise the
same underlying `Catalog` calls the hard way). Unlike `docs/notebooks/catalog_session_example.ipynb`
(a documentation walkthrough), this one exists to actually **prove** the behaviour: every
`%catalog` call below is followed by an independent check, through a fresh `Catalog` client the
magic never touched, that the change already landed — no separate `%catalog commit` anywhere in
this notebook.

**This writes to the catalog for real** (a scratch folder, cleaned up at the end — see
"Cleanup"). Run from this project's own dev environment (`pip install -e ".[dev]"`), which already
includes the `notebook` extra; `debugger/.env` must be filled in (copy from `.env.example`) and
the stack it points at reachable.

In [ ]:
import os
from pathlib import Path


def _find_debugger_env(start: Path | None = None) -> Path:
    """`debugger/.env` — located by walking up from this notebook rather than
    hardcoded, so it works wherever Jupyter's cwd happens to be (same
    reasoning as `03_ingest.ipynb`'s `_repo_root()`). Identified by sitting
    next to `debug_run.py`, not just any `.env`."""
    here = start or Path.cwd()
    for candidate in (here, *here.parents):
        if (candidate / "debug_run.py").exists() and (candidate / ".env").exists():
            return candidate / ".env"
    raise RuntimeError(
        f"could not find debugger/.env from {here} — copy debugger/.env.example to "
        "debugger/.env, fill it in, and open this notebook from inside the repository"
    )


ENV_PATH = _find_debugger_env()
for _line in ENV_PATH.read_text().splitlines():
    _line = _line.strip()
    if not _line or _line.startswith("#") or "=" not in _line:
        continue
    _key, _, _value = _line.partition("=")
    os.environ.setdefault(_key.strip(), _value.strip())

# %catalog reads DREMIO_USERNAME (see notebook/magics.py's _build_catalog_session);
# debugger/.env uses the shorter DREMIO_USER — same bridging debug_run.py itself
# does for the DDS side (_DREMIO_USER/_DREMIO_PWD).
os.environ.setdefault("DREMIO_USERNAME", os.environ.get("DREMIO_USER", ""))

print(f"env file         {ENV_PATH}")
print(f"DREMIO_BASE_URL  {os.environ.get('DREMIO_BASE_URL') or '<unset>'}")
print(f"DREMIO_USERNAME  {os.environ.get('DREMIO_USERNAME') or '<unset>'}")
print(f"DREMIO_TOKEN     {'set' if os.environ.get('DREMIO_TOKEN') else '<unset>'}")


In [1]:
import eea_datalakehouse.notebook  # registers %catalog/%ingest — no %load_ext needed

## Help

Sanity check before anything else: `%catalog help()` should print the command table without
needing any of the environment above (it's answered before a session is built).

In [ ]:
%catalog help()

## Scratch space

Everything below happens under a folder distinctly named for this notebook, alongside the
`altia_test` scratch folder `debug_run.py` already uses for the same kind of manual testing —
so it's obviously not real data, and `delete_folder(cascade=True)` in "Cleanup" removes all of
it at the end.

In [ ]:
TEST_ROOT = "catalog.water_management_resources.bathing_water.bwd.draft"
TEST_FOLDER = f"{TEST_ROOT}.catalog_magic_smoke_test"
print(TEST_FOLDER)

## 1. A call runs immediately

`create_folder` below is the only `%catalog` call in this cell — nothing else queues it, and
there's no `%catalog commit` after it.

In [ ]:
%catalog create_folder(TEST_FOLDER, create_parents=True)

**Proof:** the next cell builds a brand new `Catalog` client — one the magic's
`CatalogSession` never touched — and asks Dremio directly whether the folder is there. If
`%catalog` only queued the step, this would say `False`.

In [ ]:
from eea_datalakehouse.catalog import Catalog

verify_catalog = Catalog(
    os.environ["DREMIO_BASE_URL"], os.environ["DREMIO_TOKEN"],
    username=os.environ.get("DREMIO_USERNAME"),
)
# `_catalog_rest.exists` is the same private check `CatalogSession.create_folder`'s own
# `run()` uses internally (see session.py) — reached into directly here only because this
# is a debug/verification notebook, same spirit as debug_run.py's `catalog._flight_executor`
# access elsewhere in this folder.
print("folder exists right now:", verify_catalog._catalog_rest.exists(TEST_FOLDER))  # noqa

## 2. Relative paths still work

`create_folder` sets the session's context to the folder it just created (not its parent —
see `CatalogSession.create_folder`'s docstring), so a following `.name` call lands *inside*
it without spelling out `TEST_FOLDER` again.

In [ ]:
%catalog create_folder(".nested")  # -> TEST_FOLDER + ".nested"

In [ ]:
print("nested folder exists right now:", verify_catalog._catalog_rest.exists(f"{TEST_FOLDER}.nested"))  # noqa

## 3. Wiki text — also immediate

`set_wiki` works on any catalog entity, folders included. Commits itself the moment its
cell runs, exactly like `create_folder` above.

In [ ]:
%catalog set_wiki(TEST_FOLDER, "# Catalog magic smoke test\n\nCreated by the %catalog debugger notebook.")

## 4. Read-only queries answer immediately too

`get_wiki`/`get_tags`/`list`/`schema` are read-only `CatalogSession` methods — no `%catalog
commit` involved, same as every other call in this notebook. `get_wiki` reads back the wiki
just set above; `list` finds no tables/views here since `TEST_FOLDER` only holds folders —
an empty list, not an error (it would only raise if `TEST_FOLDER` itself didn't exist).
`list`'s own `path` can also be omitted entirely, to list whatever the current context is —
the next cell does exactly that, having just pointed context at `TEST_FOLDER` via `use`. The
cell after that passes `TEST_ROOT` — a `catalog.`-prefixed path — while context is still
`TEST_FOLDER`, to prove it's taken literally as absolute rather than appended to it (it
would otherwise become `TEST_FOLDER.catalog....`, which doesn't exist, and raise).

In [ ]:
print(verify_catalog.getwikifrom(TEST_FOLDER, idempotency_key="smoke-test-read-wiki"))

In [ ]:
%catalog list(TEST_FOLDER)  # -> [] (only folders live here — not an error, unlike a typo'd path)

In [ ]:
%catalog use(TEST_FOLDER)
%catalog list()  # path omitted — lists the current context (TEST_FOLDER) itself; same [] result

In [ ]:
%catalog list(TEST_ROOT)  # starts with 'catalog' — always absolute, never appended to
                          # the current context (still TEST_FOLDER, per the cell above)

## 5. What a failed call looks like

`TEST_FOLDER` still has `.nested` inside it, so deleting it without `cascade=True` should
fail — `%catalog` prints a short message instead of a traceback (see the design doc's
"Exceptions translated at the boundary").

In [ ]:
%catalog delete_folder(TEST_FOLDER)  # cascade=False (the default) — expected to fail, folder not empty

## Cleanup

`cascade=True` removes `.nested` along with `TEST_FOLDER` itself — this, like `%catalog`'s
other calls, has already happened by the time the cell below finishes.

In [ ]:
%catalog delete_folder(TEST_FOLDER, cascade=True)

In [ ]:
print("folder exists after cleanup:", verify_catalog._catalog_rest.exists(TEST_FOLDER))  # noqa